# Train MLP Cases bằng PyTorch

Notebook này đọc lại dữ liệu đã chuẩn hóa từ `EDASolarPowerGenerationData` / `prepare_train_test_data.py`, sau đó huấn luyện 9 cấu hình MLP theo bảng:

- Hidden layer 1 = `a`
- Hidden layer 2 = `1, 3, 6`
- Hidden layer 3 = `1, 3, 6`

Với bài toán hồi quy dự đoán `AC_POWER_TOTAL`, số neuron output là `1`.

In [1]:
from __future__ import annotations

import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
TRAIN_PATH = OUTPUT_DIR / "train_data_normalized.csv"
TEST_PATH = OUTPUT_DIR / "test_data_normalized.csv"
METADATA_PATH = OUTPUT_DIR / "train_test_metadata.json"

TARGET_COLUMN = "AC_POWER_TOTAL"
ID_COLUMNS = {"PLANT_ID", "DATE_TIME", "PLANT_NO"}
RANDOM_STATE = 42

torch.__version__

'2.5.1+cu121'

## Thiết lập seed và device

In [2]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Đọc dữ liệu train/test từ output của EDA

In [3]:
if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        "Không tìm thấy outputs/train_data_normalized.csv hoặc outputs/test_data_normalized.csv. "
        "Hãy chạy notebook EDASolarPowerGenerationData hoặc prepare_train_test_data.py trước."
    )

train_df = pd.read_csv(TRAIN_PATH, parse_dates=["DATE_TIME"])
test_df = pd.read_csv(TEST_PATH, parse_dates=["DATE_TIME"])
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8")) if METADATA_PATH.exists() else {}

train_df.shape, test_df.shape

((5133, 32), (1284, 32))

In [4]:
def get_feature_columns(df: pd.DataFrame, metadata: dict) -> list[str]:
    metadata_features = metadata.get("numeric_feature_columns_scaled", [])
    plant_features = metadata.get("plant_one_hot_columns", [])
    feature_columns = [col for col in [*metadata_features, *plant_features] if col in df.columns]

    if feature_columns:
        return feature_columns

    return [
        col
        for col in df.columns
        if col not in ID_COLUMNS and col != TARGET_COLUMN and pd.api.types.is_numeric_dtype(df[col])
    ]


feature_columns = get_feature_columns(train_df, metadata)
input_count = len(feature_columns)
output_count = 1
a = math.ceil((input_count + output_count) / 2)

print("Số thuộc tính đầu vào:", input_count)
print("Số đầu ra:", output_count)
print(f"a = ceil(({input_count} + {output_count}) / 2) = {a}")
feature_columns

Số thuộc tính đầu vào: 28
Số đầu ra: 1
a = ceil((28 + 1) / 2) = 15


['AMBIENT_TEMPERATURE',
 'MODULE_TEMPERATURE',
 'IRRADIATION',
 'HOUR',
 'MINUTE',
 'DAYOFWEEK',
 'DAYOFYEAR',
 'MONTH',
 'HOUR_SIN',
 'HOUR_COS',
 'IS_DAYLIGHT',
 'OM_TEMPERATURE_2M',
 'OM_RELATIVE_HUMIDITY_2M',
 'OM_DEW_POINT_2M',
 'OM_APPARENT_TEMPERATURE',
 'OM_PRESSURE_MSL',
 'OM_SURFACE_PRESSURE',
 'OM_PRECIPITATION',
 'OM_CLOUD_COVER',
 'OM_WIND_SPEED_10M',
 'OM_WIND_DIRECTION_10M',
 'OM_SHORTWAVE_RADIATION',
 'OM_DIRECT_RADIATION',
 'OM_DIFFUSE_RADIATION',
 'OM_DIRECT_NORMAL_IRRADIANCE',
 'OM_SUNSHINE_DURATION',
 'PLANT_NO_1',
 'PLANT_NO_2']

## Chuẩn bị tensor

`X_train` đã được chuẩn hóa từ pipeline EDA. Target `AC_POWER_TOTAL` vẫn ở scale gốc, nên notebook scale riêng `y_train` trong lúc train rồi inverse-transform dự đoán về scale gốc để tính MAE, MSE, RMSE, R2.

In [5]:
X_train_np = train_df[feature_columns].to_numpy(dtype=np.float32)
X_test_np = test_df[feature_columns].to_numpy(dtype=np.float32)
y_train_np = train_df[TARGET_COLUMN].to_numpy(dtype=np.float32).reshape(-1, 1)
y_test_np = test_df[TARGET_COLUMN].to_numpy(dtype=np.float32).reshape(-1, 1)

y_scaler = StandardScaler()
y_train_scaled_np = y_scaler.fit_transform(y_train_np).astype(np.float32)

X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled_np, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

X_train_np.shape, y_train_np.shape, X_test_np.shape, y_test_np.shape

((5133, 28), (5133, 1), (1284, 28), (1284, 1))

## Khai báo MLP và các metric

In [6]:
class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim: int, hidden_layer_sizes: tuple[int, int, int]):
        super().__init__()
        h1, h2, h3 = hidden_layer_sizes
        self.network = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, h3),
            nn.ReLU(),
            nn.Linear(h3, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MSE": float(mse),
        "RMSE": float(math.sqrt(mse)),
        "R2": float(r2_score(y_true, y_pred)),
    }


def build_case_architectures(first_layer_neurons: int) -> list[dict]:
    layer_values = [(1, 1), (1, 3), (1, 6), (3, 1), (3, 3), (3, 6), (6, 1), (6, 3), (6, 6)]
    return [
        {"case": case_no, "hidden_layer_sizes": (first_layer_neurons, second_layer, third_layer)}
        for case_no, (second_layer, third_layer) in enumerate(layer_values, start=1)
    ]


architectures = build_case_architectures(a)
architectures

[{'case': 1, 'hidden_layer_sizes': (15, 1, 1)},
 {'case': 2, 'hidden_layer_sizes': (15, 1, 3)},
 {'case': 3, 'hidden_layer_sizes': (15, 1, 6)},
 {'case': 4, 'hidden_layer_sizes': (15, 3, 1)},
 {'case': 5, 'hidden_layer_sizes': (15, 3, 3)},
 {'case': 6, 'hidden_layer_sizes': (15, 3, 6)},
 {'case': 7, 'hidden_layer_sizes': (15, 6, 1)},
 {'case': 8, 'hidden_layer_sizes': (15, 6, 3)},
 {'case': 9, 'hidden_layer_sizes': (15, 6, 6)}]

## Train 9 case MLP

In [7]:
def train_one_case(
    hidden_layer_sizes: tuple[int, int, int],
    epochs: int = 500,
    learning_rate: float = 0.001,
    weight_decay: float = 0.0001,
) -> tuple[np.ndarray, list[float]]:
    set_seed(RANDOM_STATE)
    model = MLPRegressorTorch(input_dim=input_count, hidden_layer_sizes=hidden_layer_sizes).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_history = []

    model.train()
    for epoch in range(epochs):
        batch_losses = []
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        loss_history.append(float(np.mean(batch_losses)))

    model.eval()
    with torch.no_grad():
        y_pred_scaled = model(X_test_tensor).detach().cpu().numpy()

    y_pred = y_scaler.inverse_transform(y_pred_scaled).ravel()
    return y_pred, loss_history


metric_rows = []
prediction_df = test_df[["DATE_TIME", "PLANT_NO", TARGET_COLUMN]].copy()
loss_histories = {}

for case in architectures:
    case_no = case["case"]
    hidden_layers = case["hidden_layer_sizes"]
    print(f"Training case {case_no}: {hidden_layers}")

    y_pred, loss_history = train_one_case(hidden_layers)
    prediction_df[f"CASE_{case_no}_PREDICTED_AC_POWER"] = y_pred
    loss_histories[case_no] = loss_history

    base_row = {
        "case": case_no,
        "hidden_layer_1": hidden_layers[0],
        "hidden_layer_2": hidden_layers[1],
        "hidden_layer_3": hidden_layers[2],
        "scope": "overall",
        "epochs": len(loss_history),
        "final_train_loss_scaled_target": loss_history[-1],
    }
    metric_rows.append({**base_row, **compute_metrics(y_test_np.ravel(), y_pred)})

    for plant_no in sorted(test_df["PLANT_NO"].dropna().unique()):
        mask = (test_df["PLANT_NO"] == plant_no).to_numpy()
        plant_row = {**base_row, "scope": f"plant_{int(plant_no)}"}
        metric_rows.append({**plant_row, **compute_metrics(y_test_np.ravel()[mask], y_pred[mask])})

metrics_df = pd.DataFrame(metric_rows)
overall_df = metrics_df[metrics_df["scope"] == "overall"].copy()
overall_df["rank_by_rmse"] = overall_df["RMSE"].rank(method="dense").astype(int)
metrics_df = metrics_df.merge(overall_df[["case", "rank_by_rmse"]], on="case", how="left")
metrics_df = metrics_df.sort_values(["rank_by_rmse", "case", "scope"]).reset_index(drop=True)

overall_df.sort_values("RMSE")[["case", "hidden_layer_1", "hidden_layer_2", "hidden_layer_3", "MAE", "MSE", "RMSE", "R2"]]

Training case 1: (15, 1, 1)
Training case 2: (15, 1, 3)
Training case 3: (15, 1, 6)
Training case 4: (15, 3, 1)
Training case 5: (15, 3, 3)
Training case 6: (15, 3, 6)
Training case 7: (15, 6, 1)
Training case 8: (15, 6, 3)
Training case 9: (15, 6, 6)


,case,hidden_layer_1,hidden_layer_2,hidden_layer_3,MAE,MSE,RMSE,R2
18,7,15,6,1,1021.121094,2986382.25,1728.115231,0.936924
21,8,15,6,3,872.524231,3373718.75,1836.768562,0.928743
9,4,15,3,1,951.914124,3552849.00,1884.900263,0.924959
3,2,15,1,3,1133.839722,4748101.00,2179.013768,0.899714
15,6,15,3,6,1338.614136,6290186.50,2508.024422,0.867143
12,5,15,3,3,1187.671387,8839347.00,2973.103934,0.813302
6,3,15,1,6,1933.856323,11414661.00,3378.559012,0.758908
24,9,15,6,6,6040.293457,47616964.00,6900.504619,-0.005729
0,1,15,1,1,6040.510742,47617820.00,6900.566643,-0.005747


## Metric theo từng plant

In [8]:
metrics_df[metrics_df["scope"] != "overall"].sort_values(["case", "scope"])[
    ["case", "scope", "hidden_layer_1", "hidden_layer_2", "hidden_layer_3", "MAE", "MSE", "RMSE", "R2"]
]

,case,scope,hidden_layer_1,hidden_layer_2,hidden_layer_3,MAE,MSE,RMSE,R2
25,1,plant_1,15,1,1,6832.039551,6.275218e+07,7921.627106,-0.001915
26,1,plant_2,15,1,1,5251.444824,3.253053e+07,5703.553980,-0.062846
10,2,plant_1,15,1,3,522.942139,8.311372e+05,911.667291,0.986730
11,2,plant_2,15,1,3,1742.837280,8.652881e+06,2941.577978,0.717291
19,3,plant_1,15,1,6,1454.473145,5.909868e+06,2431.022110,0.905642
20,3,plant_2,15,1,6,2411.748047,1.690233e+07,4111.244337,0.447762
7,4,plant_1,15,3,1,651.191956,1.014092e+06,1007.021381,0.983809
8,4,plant_2,15,3,1,1251.701050,6.083709e+06,2466.517586,0.801231
16,5,plant_1,15,3,3,472.311707,6.756694e+05,821.991104,0.989212
17,5,plant_2,15,3,3,1900.805786,1.697763e+07,4120.392214,0.445302


## Lưu kết quả

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_path = OUTPUT_DIR / "pytorch_mlp_case_metrics.csv"
predictions_path = OUTPUT_DIR / "pytorch_mlp_case_predictions.csv"
loss_path = OUTPUT_DIR / "pytorch_mlp_loss_history.json"

metrics_df.to_csv(metrics_path, index=False)
prediction_df.to_csv(predictions_path, index=False)
loss_path.write_text(json.dumps({str(k): v for k, v in loss_histories.items()}, indent=2), encoding="utf-8")

print("Đã lưu metrics:", metrics_path)
print("Đã lưu predictions:", predictions_path)
print("Đã lưu loss history:", loss_path)

Đã lưu metrics: z:\DOCUMENTS\KPDL\Prj_gr13\outputs\pytorch_mlp_case_metrics.csv
Đã lưu predictions: z:\DOCUMENTS\KPDL\Prj_gr13\outputs\pytorch_mlp_case_predictions.csv
Đã lưu loss history: z:\DOCUMENTS\KPDL\Prj_gr13\outputs\pytorch_mlp_loss_history.json


## Case tốt nhất

In [10]:
best_case = overall_df.sort_values("RMSE").iloc[0]
print("Best case by RMSE:", int(best_case["case"]))
print("Architecture:", int(best_case["hidden_layer_1"]), int(best_case["hidden_layer_2"]), int(best_case["hidden_layer_3"]))
print("MAE:", best_case["MAE"])
print("RMSE:", best_case["RMSE"])
print("R2:", best_case["R2"])

Best case by RMSE: 7
Architecture: 15 6 1
MAE: 1021.12109375
RMSE: 1728.1152305329642
R2: 0.9369239211082458
